# FinWizard

Stock-price experiments using Yahoo Finance data, news sentiment, and LSTM, CNN, and RNN models.

See the repository README for dependencies, required data, and current limitations.


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from sklearn.preprocessing import RobustScaler
import requests

# Initialize VADER analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get stock data
def get_stock_data(ticker, start_date, end_date):
    try:
        stock_data = yf.download(ticker, start=start_date, end=end_date)
        if stock_data.empty:
            print(f"No stock data available for {ticker}")
            return None
        stock_data.reset_index(inplace=True)
        stock_data = stock_data[['Date', 'Close', 'Open', 'High', 'Low', 'Volume']]
        stock_data.columns = ['date', 'price', 'open', 'high', 'low', 'volume']
        return stock_data
    except Exception as e:
        print(f"Error fetching stock data for {ticker}: {e}")
        return None

# Function to get news data and analyze sentiment
def get_news_data_with_sentiment(query, from_date, to_date, api_key):
    try:
        url = f"https://newsapi.org/v2/everything?q={query}&from={from_date}&to={to_date}&sortBy=popularity&apiKey={api_key}"
        response = requests.get(url)
        articles = response.json().get("articles", [])

        if not articles:
            print(f"No news articles found for {query}")
            return None

        news_data = pd.DataFrame({
            "date": [article["publishedAt"][:10] for article in articles],
            "headline": [article["title"] for article in articles]
        })

        news_data['sentiment'] = news_data['headline'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
        news_data['date'] = pd.to_datetime(news_data['date'])

        daily_sentiment = news_data.groupby('date').mean().reset_index()
        return daily_sentiment
    except Exception as e:
        print(f"Error fetching news data for {query}: {e}")
        return None

# Function for modeling and predictions
def stock_price_prediction(ticker, query, start_date, end_date, api_key, last_month_date):
    stock_data = get_stock_data(ticker, start_date, end_date)
    if stock_data is None:
        return None

    news_sentiment_data = get_news_data_with_sentiment(query, start_date, end_date, api_key)
    if news_sentiment_data is None:
        return None

    stock_data['date'] = pd.to_datetime(stock_data['date']).dt.tz_localize(None)
    news_sentiment_data['date'] = pd.to_datetime(news_sentiment_data['date']).dt.tz_localize(None)

    # Merge stock and sentiment data
    merged_data = pd.merge(stock_data, news_sentiment_data, on='date', how='left')
    merged_data.fillna(0, inplace=True)

    if merged_data.empty:
        print(f"No merged data available for {ticker}")
        return None

    scaler = RobustScaler()
    scaled_data = scaler.fit_transform(merged_data[['price', 'open', 'high', 'low', 'volume', 'sentiment']])

    # Create sequences
    look_back = 60
    X, y = [], []
    for i in range(look_back, len(scaled_data)):
        X.append(scaled_data[i - look_back:i])
        y.append(scaled_data[i, 0])
    X, y = np.array(X), np.array(y)

    train_size = int(0.8 * len(X))
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    # Define LSTM model
    def lstm_model():
        model = Sequential([
            LSTM(100, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
            Dropout(0.3),
            LSTM(50),
            Dropout(0.3),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mean_absolute_error')
        return model

    lstm = lstm_model()
    lstm.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=1)

    # Generate predictions
    predictions = lstm.predict(X_test)

    # Inverse scale predictions
    def inverse_transform(predictions):
        return scaler.inverse_transform(np.concatenate((predictions, np.zeros((predictions.shape[0], 5))), axis=1))[:, 0]

    actual_prices = inverse_transform(y_test.reshape(-1, 1))
    predicted_prices = inverse_transform(predictions)

    # Last month data
    last_month_data = merged_data[merged_data['date'] >= last_month_date]
    if last_month_data.empty:
        print(f"No last month data for {ticker}")
        return None

    last_month_actual = last_month_data['price'].values
    last_month_predicted = predicted_prices[-len(last_month_actual):]
    last_month_dates = last_month_data['date'].dt.date.values

    result = pd.DataFrame({
        'Date': last_month_dates,
        'Actual Price': last_month_actual,
        'Predicted Price': last_month_predicted
    })

    return result

# Define stocks (tickers only)
banks = [
    'SUNPHARMA.NS', 'ALKEM.NS', 'LUPIN.NS', 'AUROPHARMA.NS', 'AJANTPHARM.NS',
    'GLENMARK.NS', 'SUVENPHAR.NS', 'NEULANDLAB.NS', 'MARKSANS.NS'
]

start_date = '2023-11-01'
end_date = '2023-12-01'
last_month_date = '2023-11-01'
api_key = __import__("os").environ["NEWS_API_KEY"]

# Collect predictions
results = []
for bank in banks:
    print(f"\nPredicting for bank: {bank}")
    result = stock_price_prediction(bank, bank, start_date, end_date, api_key, last_month_date)
    if result is not None:
        result['Pharmaceuticals'] = bank
        results.append(result)

if results:
    final_result = pd.concat(results, ignore_index=True)
    final_result.to_excel('Pharmaceuticals_Stock_Predictions.xlsx', index=False)
    print("Predictions saved to 'Pharmaceuticals_Stock_Predictions.xlsx'")
else:
    print("No predictions were generated.")


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error

# Function to fetch stock data
def fetch_stock_data(ticker, period='6mo', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data

# Function to calculate moving averages
def calculate_moving_averages(data, short_window=20, long_window=50):
    data['Short_MA'] = data['Close'].rolling(window=short_window).mean()
    data['Long_MA'] = data['Close'].rolling(window=long_window).mean()
    return data

# Function to generate buy/sell signals
def generate_trading_signals(data):
    data['Signal'] = 0  # Default: No action
    data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1  # Buy
    data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1  # Sell
    return data

# Function to prepare data for LSTM
def prepare_lstm_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Function to build LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Function to make predictions
def make_predictions(model, X, scaler):
    predicted = model.predict(X)
    return scaler.inverse_transform(predicted)

# Function to calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Fetch stock data for ICICI Bank
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Calculate moving averages and trading signals
stock_data = calculate_moving_averages(stock_data)
stock_data = generate_trading_signals(stock_data)

# Prepare data for LSTM model
X_train, y_train, scaler = prepare_lstm_data(stock_data)
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Predict stock prices for the same time period
predicted_prices = make_predictions(model, X_train, scaler)

# Add predictions to DataFrame
stock_data = stock_data.iloc[-len(predicted_prices):]  # Align lengths
stock_data['Predicted_Close'] = predicted_prices.flatten()

# Calculate Model Accuracy (MSE, RMSE, MAPE)
mse = mean_squared_error(stock_data['Close'], stock_data['Predicted_Close'])
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(stock_data['Close'].values, stock_data['Predicted_Close'].values)

print(f"\n📊 Model Accuracy:")
print(f"🔹 Mean Squared Error (MSE): {mse:.4f}")
print(f"🔹 Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"🔹 Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

# Select last 30 days for text output
past_30_days = stock_data.iloc[-30:][['Close', 'Predicted_Close', 'Signal']]
past_30_days['Recommendation'] = past_30_days['Signal'].map({1: 'BUY', -1: 'SELL', 0: 'HOLD'})

# Display past 30 days' actual & predicted stock data with recommendations
print("\n📊 Past 30 Days Stock Data with Buy/Sell Signals & Predicted Prices:\n")
print(past_30_days[['Close', 'Predicted_Close', 'Recommendation']].to_string(index=True))

# Plot the stock prices and predictions
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Close'],
    mode='lines',
    name='Actual Price',
    line=dict(color='white', width=2)
))

# Plot predicted prices
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Predicted_Close'],
    mode='lines',
    name='Predicted Price (LSTM)',
    line=dict(color='yellow', dash='dot', width=2)
))

# Plot short-term moving average
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Short_MA'],
    mode='lines',
    name='20-Day SMA',
    line=dict(color='cyan', dash='dot', width=2)
))

# Plot long-term moving average
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Long_MA'],
    mode='lines',
    name='50-Day SMA',
    line=dict(color='magenta', dash='dot', width=2)
))

# Plot buy signals
buy_signals = stock_data[stock_data['Signal'] == 1]
fig.add_trace(go.Scatter(
    x=buy_signals.index,
    y=buy_signals['Close'],
    mode='markers',
    name='Buy Signal',
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals
sell_signals = stock_data[stock_data['Signal'] == -1]
fig.add_trace(go.Scatter(
    x=sell_signals.index,
    y=sell_signals['Close'],
    mode='markers',
    name='Sell Signal',
    marker=dict(color='red', size=10, symbol='triangle-down')
))

# Update graph layout
fig.update_layout(
    title='📈 ICICI Bank Stock Price with LSTM Predictions & Buy/Sell Signals',
    xaxis_title='📅 Date',
    yaxis_title='💰 Stock Price (INR)',
    template='plotly_dark',
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

# Show the graph
fig.show()



In [ ]:
# Chumeshwari
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error

# Fetch Stock Data
def fetch_stock_data(ticker, period='1y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data

# Calculate Moving Averages
def calculate_moving_averages(data, short_window=20, long_window=50):
    data['Short_MA'] = data['Close'].rolling(window=short_window).mean()
    data['Long_MA'] = data['Close'].rolling(window=long_window).mean()
    return data

# Generate Trading Signals
def generate_trading_signals(data):
    data['Signal'] = 0
    data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1
    data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1
    return data

# Prepare Data for LSTM
def prepare_lstm_data(data, n_steps=60):
    data['Log_Close'] = np.log1p(data['Close'])  # Apply log transformation
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Log_Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i - n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build LSTM Model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),  # Prevent overfitting
        LSTM(100),
        Dropout(0.2),
        Dense(50, activation='relu'),  # Extra Dense Layer
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001, decay=1e-6), loss='mean_squared_error')
    return model

# Make Predictions
def make_predictions(model, X, scaler):
    predicted = model.predict(X)
    predicted = np.expm1(scaler.inverse_transform(predicted))  # Inverse log transform
    return predicted

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Fetch stock data
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Apply moving averages and signals
stock_data = calculate_moving_averages(stock_data)
stock_data = generate_trading_signals(stock_data)

# Prepare data
X_train, y_train, scaler = prepare_lstm_data(stock_data)

# Train Model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.1, verbose=1)

# Make Predictions
predicted_prices = make_predictions(model, X_train, scaler)

# Align DataFrame Lengths
stock_data = stock_data.iloc[-len(predicted_prices):]
stock_data['Predicted_Close'] = predicted_prices.flatten()

# Calculate Model Accuracy
mse = mean_squared_error(stock_data['Close'], stock_data['Predicted_Close'])
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(stock_data['Close'].values, stock_data['Predicted_Close'].values)

print(f"\n📊 Model Accuracy:")
print(f"🔹 Mean Squared Error (MSE): {mse:.4f}")
print(f"🔹 Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"🔹 Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

# Get last 30 days' predictions
past_30_days = stock_data.iloc[-30:][['Close', 'Predicted_Close', 'Signal']]
past_30_days['Recommendation'] = past_30_days['Signal'].map({1: 'BUY', -1: 'SELL', 0: 'HOLD'})

# Display past data
print("\n📊 Past 30 Days Stock Data with Buy/Sell Signals & Predicted Prices:\n")
print(past_30_days[['Close', 'Predicted_Close', 'Recommendation']].to_string(index=True))

# -----------------------------
# 📈 Research Paper Style Plot
# -----------------------------
fig = go.Figure()

# Actual Prices
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Close'],
    mode='lines',
    name='Actual Price',
    line=dict(color='black', width=2)
))

# Predicted Prices
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Predicted_Close'],
    mode='lines',
    name='Predicted Price (LSTM)',
    line=dict(color='royalblue', width=2, dash='dot')
))

# Moving Averages
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Short_MA'],
    mode='lines',
    name='20-Day SMA',
    line=dict(color='orange', width=1.5, dash='dash')
))

fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Long_MA'],
    mode='lines',
    name='50-Day SMA',
    line=dict(color='green', width=1.5, dash='dash')
))

# Buy Signals
buy_signals = stock_data[stock_data['Signal'] == 1]
fig.add_trace(go.Scatter(
    x=buy_signals.index,
    y=buy_signals['Close'],
    mode='markers',
    name='Buy Signal',
    marker=dict(color='darkgreen', size=8, symbol='triangle-up')
))

# Sell Signals
sell_signals = stock_data[stock_data['Signal'] == -1]
fig.add_trace(go.Scatter(
    x=sell_signals.index,
    y=sell_signals['Close'],
    mode='markers',
    name='Sell Signal',
    marker=dict(color='firebrick', size=8, symbol='triangle-down')
))

# Graph Layout
fig.update_layout(
    title='ICICI Bank Stock Price: LSTM Predictions & Trading Signals',
    xaxis_title='Date',
    yaxis_title='Stock Price (INR)',
    template='plotly_white',
    font=dict(family='Arial', size=14, color='black'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    height=600,
    width=1000
)

# Show Graph
fig.show()


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error

# Stock tickers from different sectors
stocks = {
    "FMCG": "HINDUNILVR.NS",
    "IT Services": "INFY.NS",
    "Automobile": "TATAMOTORS.NS",
    "Health Care": "SUNPHARMA.NS",
    "Energy": "ONGC.NS"
}

# Fetch Stock Data
def fetch_stock_data(ticker, period='1y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data

# Calculate Moving Averages
def calculate_moving_averages(data, short_window=20, long_window=50):
    data['Short_MA'] = data['Close'].rolling(window=short_window).mean()
    data['Long_MA'] = data['Close'].rolling(window=long_window).mean()
    return data

# Generate Trading Signals
def generate_trading_signals(data):
    data['Signal'] = 0
    data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1
    data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1
    return data

# Prepare Data for LSTM
def prepare_lstm_data(data, n_steps=60):
    data['Log_Close'] = np.log1p(data['Close'])  # Log transform
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Log_Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i - n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X = np.array(X)
    y = np.array(y)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    return X, y, scaler

# Build LSTM Model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(100),
        Dropout(0.2),
        Dense(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001, decay=1e-6), loss='mean_squared_error')
    return model

# Make Predictions
def make_predictions(model, X, scaler):
    predicted = model.predict(X)
    predicted = np.expm1(scaler.inverse_transform(predicted))  # Inverse log transform
    return predicted

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Train and Predict for Each Stock
results = {}

for sector, ticker in stocks.items():
    print(f"\n📈 Training LSTM Model for {sector} Sector ({ticker})...\n")

    stock_data = fetch_stock_data(ticker)
    stock_data = calculate_moving_averages(stock_data)
    stock_data = generate_trading_signals(stock_data)

    X_train, y_train, scaler = prepare_lstm_data(stock_data)

    model = build_lstm_model((X_train.shape[1], 1))
    model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.1, verbose=1)

    predicted_prices = make_predictions(model, X_train, scaler)

    stock_data = stock_data.iloc[-len(predicted_prices):]
    stock_data['Predicted_Close'] = predicted_prices.flatten()

    mse = mean_squared_error(stock_data['Close'], stock_data['Predicted_Close'])
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(stock_data['Close'].values, stock_data['Predicted_Close'].values)

    print(f"\n📊 {sector} Sector ({ticker}) Model Accuracy:")
    print(f"🔹 MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAPE: {mape:.2f}%")

    past_30_days = stock_data.iloc[-30:][['Close', 'Predicted_Close', 'Signal']]
    past_30_days['Recommendation'] = past_30_days['Signal'].map({1: 'BUY', -1: 'SELL', 0: 'HOLD'})

    results[sector] = {"Stock": ticker, "Data": past_30_days}

    # ----------------------------
    # 📈 Enhanced Graph for Paper
    # ----------------------------
    fig = go.Figure()

    # Actual Prices
    fig.add_trace(go.Scatter(
        x=stock_data.index,
        y=stock_data['Close'],
        mode='lines',
        name='Actual Price',
        line=dict(color='black', width=2)
    ))

    # Predicted Prices
    fig.add_trace(go.Scatter(
        x=stock_data.index,
        y=stock_data['Predicted_Close'],
        mode='lines',
        name='Predicted Price (LSTM)',
        line=dict(color='royalblue', width=2, dash='dot')
    ))

    # Moving Averages
    fig.add_trace(go.Scatter(
        x=stock_data.index,
        y=stock_data['Short_MA'],
        mode='lines',
        name='20-Day SMA',
        line=dict(color='orange', dash='dash', width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=stock_data.index,
        y=stock_data['Long_MA'],
        mode='lines',
        name='50-Day SMA',
        line=dict(color='green', dash='dash', width=1.5)
    ))

    # Buy/Sell Markers
    buy = stock_data[stock_data['Signal'] == 1]
    sell = stock_data[stock_data['Signal'] == -1]

    fig.add_trace(go.Scatter(
        x=buy.index,
        y=buy['Close'],
        mode='markers',
        name='Buy Signal',
        marker=dict(color='darkgreen', size=8, symbol='triangle-up')
    ))

    fig.add_trace(go.Scatter(
        x=sell.index,
        y=sell['Close'],
        mode='markers',
        name='Sell Signal',
        marker=dict(color='firebrick', size=8, symbol='triangle-down')
    ))

    # Layout
    fig.update_layout(
        title=f'{sector} Sector: {ticker} LSTM Prediction & Trading Signals',
        xaxis_title='Date',
        yaxis_title='Price (INR)',
        template='plotly_white',
        font=dict(family='Arial', size=14, color='black'),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        margin=dict(l=40, r=40, t=80, b=40),
        height=600,
        width=1000
    )

    fig.show()

# Print Final 30-Day Summary for All
print("\n📊 Past 30 Days Stock Data with Buy/Sell Signals for All Sectors:\n")
for sector, data in results.items():
    print(f"\n📈 {sector} Sector ({data['Stock']}):")
    print(data["Data"].to_string(index=True))


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go

# Function to fetch live stock data
def fetch_live_stock_data(ticker, period='6mo', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data

# Function to calculate moving averages
def calculate_moving_averages(data, short_window=20, long_window=50):
    data['Short_MA'] = data['Close'].rolling(window=short_window).mean()
    data['Long_MA'] = data['Close'].rolling(window=long_window).mean()
    return data

# Function to generate buy/sell signals
def generate_trading_signals(data):
    data['Signal'] = 0  # Default: No action
    data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1  # Buy
    data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1  # Sell
    return data

# Fetch stock data for ICICI Bank (Indian NSE)
ticker = 'ICICIBANK.NS'
stock_data = fetch_live_stock_data(ticker)

# Calculate moving averages
stock_data = calculate_moving_averages(stock_data)

# Generate buy/sell signals
stock_data = generate_trading_signals(stock_data)

# Select the last 30 days of data for text output
past_30_days = stock_data.iloc[-30:][['Close', 'Signal']]

# Convert signals to readable text
past_30_days['Recommendation'] = past_30_days['Signal'].map({1: 'BUY', -1: 'SELL', 0: 'HOLD'})

# Display past 30 days of stock data with buy/sell recommendations
print("\n📊 Past 30 Days Stock Data with Buy/Sell Signals:\n")
print(past_30_days[['Close', 'Recommendation']].to_string(index=True))

# Create the figure
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Close'],
    mode='lines',
    name='ICICI Bank Price',
    line=dict(color='white', width=2)
))

# Plot short-term moving average (20-day SMA)
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Short_MA'],
    mode='lines',
    name='20-Day SMA',
    line=dict(color='cyan', dash='dot', width=2)
))

# Plot long-term moving average (50-day SMA)
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Long_MA'],
    mode='lines',
    name='50-Day SMA',
    line=dict(color='magenta', dash='dot', width=2)
))

# Plot buy signals (Green ▲)
buy_signals = stock_data[stock_data['Signal'] == 1]
fig.add_trace(go.Scatter(
    x=buy_signals.index,
    y=buy_signals['Close'],
    mode='markers',
    name='Buy Signal',
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals (Red ▼)
sell_signals = stock_data[stock_data['Signal'] == -1]
fig.add_trace(go.Scatter(
    x=sell_signals.index,
    y=sell_signals['Close'],
    mode='markers',
    name='Sell Signal',
    marker=dict(color='red', size=10, symbol='triangle-down')
))

# Update graph layout
fig.update_layout(
    title=dict(text='📈 ICICI Bank Stock Price with Buy/Sell Signals', font=dict(size=18)),
    xaxis_title='📅 Date',
    yaxis_title='💰 Stock Price (INR)',
    template='plotly_dark',
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

# Show the graph
fig.show()





In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta

# Function to fetch live stock data
def fetch_live_stock_data(ticker, period='6mo', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data

# Function to calculate moving averages
def calculate_moving_averages(data, short_window=20, long_window=50):
    data['Short_MA'] = data['Close'].rolling(window=short_window).mean()
    data['Long_MA'] = data['Close'].rolling(window=long_window).mean()
    return data

# Function to generate buy/sell signals based on moving average crossover strategy
def generate_trading_signals(data):
    data['Signal'] = 0  # Default: No action
    data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1  # Buy
    data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1  # Sell
    return data

# Fetch stock data for ICICI Bank (Indian NSE)
ticker = 'ICICIBANK.NS'
stock_data = fetch_live_stock_data(ticker)

# Calculate moving averages
stock_data = calculate_moving_averages(stock_data)

# Generate buy/sell signals
stock_data = generate_trading_signals(stock_data)

# Plot stock prices with buy/sell markers
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Close'],
    mode='lines',
    name='ICICI Bank Stock Price'
))

# Plot short-term moving average
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Short_MA'],
    mode='lines',
    name='Short-Term MA (20 days)',
    line=dict(color='blue', dash='dot')
))

# Plot long-term moving average
fig.add_trace(go.Scatter(
    x=stock_data.index,
    y=stock_data['Long_MA'],
    mode='lines',
    name='Long-Term MA (50 days)',
    line=dict(color='red', dash='dot')
))

# Plot buy signals
buy_signals = stock_data[stock_data['Signal'] == 1]
fig.add_trace(go.Scatter(
    x=buy_signals.index,
    y=buy_signals['Close'],
    mode='markers',
    name='Buy Signal',
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals
sell_signals = stock_data[stock_data['Signal'] == -1]
fig.add_trace(go.Scatter(
    x=sell_signals.index,
    y=sell_signals['Close'],
    mode='markers',
    name='Sell Signal',
    marker=dict(color='red', size=10, symbol='triangle-down')
))

fig.update_layout(
    title='ICICI Bank Live Stock Price with Buy/Sell Signals',
    xaxis_title='Date',
    yaxis_title='Stock Price (INR)',
    template='plotly_dark'
)

fig.show()


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Function to fetch stock data
def fetch_stock_data(ticker, period='5y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# Function to prepare data for LSTM
def prepare_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Predict future stock prices
def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    for _ in range(days_to_predict):
        pred = model.predict(np.reshape(last_steps, (1, last_steps.shape[0], 1)))
        predictions.append(pred[0, 0])
        last_steps = np.append(last_steps[1:], pred)

    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# Generate Buy/Sell Recommendations
def generate_recommendations(predictions):
    recommendations = ["Buy" if predictions[i] > predictions[i-1] else "Sell" for i in range(1, len(predictions))]
    recommendations.insert(0, "Hold")  # First day default to hold
    return recommendations

# Fetch stock data
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Prepare data
X_train, y_train, scaler = prepare_data(stock_data.values)

# Train the LSTM model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Predict future stock prices
future_days = [7, 30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# Generate Buy/Sell recommendations
recommendations = {days: generate_recommendations(future_predictions[days]) for days in future_days}

# Plot actual stock prices
fig = go.Figure()
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Close'], mode='lines', name='Actual Prices'))

# Plot predictions
colors = {'7': 'green', '30': 'blue', '365': 'red'}
for days in future_days:
    future_dates = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=days)
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=future_predictions[days].flatten(),
        mode='lines',
        name=f'Predicted Prices ({days} Days)',
        line=dict(dash='dash', color=colors[str(days)])
    ))

fig.update_layout(
    title='ICICI Bank Stock Price Prediction & Buy/Sell Recommendations',
    xaxis_title='Date',
    yaxis_title='Stock Price (INR)',
    template='plotly_dark'
)

fig.show()

# Display recommendations
for days in future_days:
    print(f"📌 Buy/Sell Recommendations for next {days} days:")
    print(recommendations[days])


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Fetch live stock data
def fetch_stock_data(ticker, period='2y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# Calculate indicators
def calculate_indicators(data):
    data['SMA_20'] = data['Close'].rolling(window=20).mean()  # Simple Moving Average
    data['EMA_50'] = data['Close'].ewm(span=50, adjust=False).mean()  # Exponential Moving Average

    # RSI Calculation
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))

    # MACD Calculation
    short_ema = data['Close'].ewm(span=12, adjust=False).mean()
    long_ema = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = short_ema - long_ema
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()

    return data

# Generate buy/sell signals
def generate_signals(data):
    data['Signal'] = 'Hold'

    # Buy when Short MA crosses above Long MA & RSI < 30 (Oversold)
    data.loc[(data['SMA_20'] > data['EMA_50']) & (data['RSI'] < 30), 'Signal'] = 'Buy'

    # Sell when Short MA crosses below Long MA & RSI > 70 (Overbought)
    data.loc[(data['SMA_20'] < data['EMA_50']) & (data['RSI'] > 70), 'Signal'] = 'Sell'

    return data

# Prepare data for LSTM
def prepare_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Predict future stock prices
def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    for _ in range(days_to_predict):
        pred = model.predict(np.reshape(last_steps, (1, last_steps.shape[0], 1)))
        predictions.append(pred[0, 0])
        last_steps = np.append(last_steps[1:], pred)

    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# Fetch stock data for ICICI Bank
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Calculate indicators & signals
stock_data = calculate_indicators(stock_data)
stock_data = generate_signals(stock_data)

# Prepare data for LSTM
X_train, y_train, scaler = prepare_data(stock_data)

# Train LSTM model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Predict future stock prices (7, 30, 365 days)
future_days = [7, 30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# Plot stock prices and indicators
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['Close'], mode='lines', name='ICICI Bank Price'
))

# Plot SMA (20-day)
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='20-Day SMA', line=dict(color='blue', dash='dot')
))

# Plot EMA (50-day)
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['EMA_50'], mode='lines', name='50-Day EMA', line=dict(color='red', dash='dot')
))

# Plot RSI
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='purple')
))

# Plot MACD
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD', line=dict(color='orange')
))
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['MACD_Signal'], mode='lines', name='MACD Signal', line=dict(color='cyan', dash='dot')
))

# Mark Buy Signals
buy_signals = stock_data[stock_data['Signal'] == 'Buy']
fig.add_trace(go.Scatter(
    x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')
))

# Mark Sell Signals
sell_signals = stock_data[stock_data['Signal'] == 'Sell']
fig.add_trace(go.Scatter(
    x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')
))

# Plot future predictions
colors = {'7': 'green', '30': 'blue', '365': 'red'}
for days in future_days:
    future_dates = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=days)
    fig.add_trace(go.Scatter(
        x=future_dates, y=future_predictions[days].flatten(),
        mode='lines', name=f'Predicted Prices ({days} Days)', line=dict(dash='dash', color=colors[str(days)])
    ))

fig.update_layout(
    title='ICICI Bank Stock Prediction with Indicators & Buy/Sell Signals',
    xaxis_title='Date', yaxis_title='Stock Price (INR)', template='plotly_dark'
)

fig.show()


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Fetch live stock data
def fetch_stock_data(ticker, period='2y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# Prepare data for LSTM
def prepare_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Predict future stock prices
def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    for _ in range(days_to_predict):
        pred = model.predict(np.reshape(last_steps, (1, last_steps.shape[0], 1)))
        predictions.append(pred[0, 0])
        last_steps = np.append(last_steps[1:], pred)

    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# Fetch stock data for ICICI Bank
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Prepare data for LSTM
X_train, y_train, scaler = prepare_data(stock_data)

# Train LSTM model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Predict future stock prices (30, 365 days)
future_days = [30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# Convert to DataFrame for better visualization
future_dates_30 = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=30)
future_dates_365 = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=365)

df_30 = pd.DataFrame({'Date': future_dates_30, 'Predicted Price': future_predictions[30].flatten()})
df_365 = pd.DataFrame({'Date': future_dates_365, 'Predicted Price': future_predictions[365].flatten()})

# Display the tables
print("Predicted Stock Prices for Next 30 Days:")
print(df_30.to_string(index=False))

print("\nPredicted Stock Prices for Next 365 Days:")
print(df_365.to_string(index=False))


In [ ]:
# Import necessary libraries
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import random
import os

# Fix the random seed for reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Fetch live stock data
def fetch_stock_data(ticker, period='5y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# Calculate technical indicators
def calculate_indicators(data):
    data['SMA_20'] = data['Close'].rolling(window=20).mean()
    data['EMA_50'] = data['Close'].ewm(span=50, adjust=False).mean()

    # RSI Calculation
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))

    # MACD Calculation
    short_ema = data['Close'].ewm(span=12, adjust=False).mean()
    long_ema = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = short_ema - long_ema
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()

    return data.dropna()

# Prepare data for LSTM
def prepare_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build optimized LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(100, return_sequences=True),
        Dropout(0.2),
        LSTM(100),
        Dense(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    last_steps = last_steps.reshape(1, last_steps.shape[0], 1)

    for _ in range(days_to_predict):
        pred = model.predict(last_steps, verbose=0)
        predictions.append(pred[0, 0])

        # Update last_steps with new prediction, reshaping pred to (1, 1, 1)
        last_steps = np.append(last_steps[:, 1:, :], pred.reshape(1, 1, 1), axis=1)

    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# Fetch stock data for ICICI Bank
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Calculate indicators & prepare data
stock_data = calculate_indicators(stock_data)
X_train, y_train, scaler = prepare_data(stock_data)

# Train LSTM model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

# Predict future stock prices (30 & 365 days)
future_days = [30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# Create DataFrames for visualization
future_dates_30 = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=30)
future_dates_365 = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=365)

df_predictions_30 = pd.DataFrame({'Date': future_dates_30, 'Predicted Price (30 Days)': future_predictions[30].flatten()})
df_predictions_365 = pd.DataFrame({'Date': future_dates_365, 'Predicted Price (365 Days)': future_predictions[365].flatten()})

# Display results in tabular format
print("\nPredicted Prices for Next 30 Days:")
print(df_predictions_30.to_string(index=False))

print("\nPredicted Prices for Next 365 Days:")
print(df_predictions_365.to_string(index=False))

# Plot stock prices and indicators
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Close'], mode='lines', name='ICICI Bank Price'))

# Plot SMA (20-day)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='20-Day SMA', line=dict(color='blue', dash='dot')))

# Plot EMA (50-day)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['EMA_50'], mode='lines', name='50-Day EMA', line=dict(color='red', dash='dot')))

# Plot RSI
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='purple')))

# Plot MACD
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD_Signal'], mode='lines', name='MACD Signal', line=dict(color='cyan', dash='dot')))

# Plot future predictions
fig.add_trace(go.Scatter(x=future_dates_30, y=future_predictions[30].flatten(), mode='lines', name='Predicted Prices (30 Days)', line=dict(dash='dash', color='blue')))
fig.add_trace(go.Scatter(x=future_dates_365, y=future_predictions[365].flatten(), mode='lines', name='Predicted Prices (365 Days)', line=dict(dash='dash', color='red')))

fig.update_layout(title='ICICI Bank Stock Prediction with Indicators', xaxis_title='Date', yaxis_title='Stock Price (INR)', template='plotly_dark')
fig.show()


In [ ]:
# Import necessary libraries
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import random
import os

# Fix the random seed for reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Fetch stock data
def fetch_stock_data(ticker, period='7y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# Calculate technical indicators
def calculate_indicators(data):
    data['SMA_20'] = data['Close'].rolling(window=20).mean()
    data['EMA_50'] = data['Close'].ewm(span=50, adjust=False).mean()

    # RSI Calculation
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))

    # MACD Calculation
    short_ema = data['Close'].ewm(span=12, adjust=False).mean()
    long_ema = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = short_ema - long_ema
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()

    # Bollinger Bands
    data['Upper_BB'] = data['SMA_20'] + (data['Close'].rolling(window=20).std() * 2)
    data['Lower_BB'] = data['SMA_20'] - (data['Close'].rolling(window=20).std() * 2)

    return data.dropna()

# Prepare data for LSTM
def prepare_data(data, n_steps=90):  # Increased steps for better learning
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build optimized LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(128),
        Dense(64, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Predict future stock prices using rolling window method
def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    last_steps = last_steps.reshape(1, last_steps.shape[0], 1)

    for _ in range(days_to_predict):
        pred = model.predict(last_steps, verbose=0)
        predictions.append(pred[0, 0])

        # Update last_steps with new prediction
        last_steps = np.append(last_steps[:, 1:, :], [[pred]], axis=1)

    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# Fetch stock data for Reliance Industries
ticker = 'RELIANCE.NS'
stock_data = fetch_stock_data(ticker)

# Calculate indicators & prepare data
stock_data = calculate_indicators(stock_data)
X_train, y_train, scaler = prepare_data(stock_data)

# Train LSTM model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
model.fit(X_train, y_train, epochs=60, batch_size=32, verbose=1)

# Predict future stock prices (30 & 365 days)
future_days = [30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# Create DataFrames for visualization
future_dates_30 = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=30)
future_dates_365 = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=365)

df_predictions_30 = pd.DataFrame({'Date': future_dates_30, 'Predicted Price (30 Days)': future_predictions[30].flatten()})
df_predictions_365 = pd.DataFrame({'Date': future_dates_365, 'Predicted Price (365 Days)': future_predictions[365].flatten()})

# Display results in tabular format
print("\nPredicted Prices for Next 30 Days:")
print(df_predictions_30.to_string(index=False))

print("\nPredicted Prices for Next 365 Days:")
print(df_predictions_365.to_string(index=False))

# Plot stock prices and indicators
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Close'], mode='lines', name='Reliance Price'))

# Plot SMA (20-day)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='20-Day SMA', line=dict(color='blue', dash='dot')))

# Plot EMA (50-day)
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['EMA_50'], mode='lines', name='50-Day EMA', line=dict(color='red', dash='dot')))

# Plot RSI
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='purple')))

# Plot MACD
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['MACD_Signal'], mode='lines', name='MACD Signal', line=dict(color='cyan', dash='dot')))

# Plot Bollinger Bands
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Upper_BB'], mode='lines', name='Upper Bollinger Band', line=dict(color='green', dash='dot')))
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['Lower_BB'], mode='lines', name='Lower Bollinger Band', line=dict(color='green', dash='dot')))

fig.update_layout(title='Reliance Stock Prediction with Indicators', xaxis_title='Date', yaxis_title='Stock Price (INR)', template='plotly_dark')
fig.show()


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# ------------------- Fetch Live Stock Data -------------------
def fetch_stock_data(ticker, period='2y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# ------------------- Calculate Indicators -------------------
def calculate_indicators(data):
    data['SMA_20'] = data['Close'].rolling(window=20).mean()  # Simple Moving Average
    data['EMA_50'] = data['Close'].ewm(span=50, adjust=False).mean()  # Exponential Moving Average

    # RSI Calculation
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))

    # MACD Calculation
    short_ema = data['Close'].ewm(span=12, adjust=False).mean()
    long_ema = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = short_ema - long_ema
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()

    return data.dropna()  # Remove NaN values after calculations

# ------------------- Generate Buy/Sell Signals -------------------
def generate_signals(data):
    data['Signal'] = 'Hold'

    # Buy when Short MA crosses above Long MA & RSI < 30 (Oversold)
    data.loc[(data['SMA_20'] > data['EMA_50']) & (data['RSI'] < 30), 'Signal'] = 'Buy'

    # Sell when Short MA crosses below Long MA & RSI > 70 (Overbought)
    data.loc[(data['SMA_20'] < data['EMA_50']) & (data['RSI'] > 70), 'Signal'] = 'Sell'

    return data

# ------------------- Prepare Data for LSTM -------------------
def prepare_data(data, n_steps=60):
    features = ['Close', 'SMA_20', 'EMA_50', 'RSI', 'MACD', 'MACD_Signal']
    data = data[features].dropna()

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i])
        y.append(scaled_data[i, 0])  # Predicting only the Close price

    X, y = np.array(X), np.array(y)
    return X, y, scaler

# ------------------- Build Optimized LSTM Model -------------------
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(100, return_sequences=False),
        Dropout(0.2),
        Dense(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# ------------------- Predict Future Stock Prices -------------------
def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    for _ in range(days_to_predict):
        pred = model.predict(np.reshape(last_steps, (1, last_steps.shape[0], last_steps.shape[1])))
        predictions.append(pred[0, 0])

        # Create next input with predicted Close price + zeros for other features
        next_input = np.zeros(last_steps.shape[1])
        next_input[0] = pred[0, 0]  # Assign predicted Close price
        last_steps = np.vstack([last_steps[1:], next_input])

    # Convert back to original scale
    predicted_prices = np.zeros((days_to_predict, scaler.n_features_in_))
    predicted_prices[:, 0] = predictions  # Set predicted Close prices

    return scaler.inverse_transform(predicted_prices)[:, 0]  # Return only Close prices

# ------------------- Fetch Stock Data & Process -------------------
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Calculate indicators & signals
stock_data = calculate_indicators(stock_data)
stock_data = generate_signals(stock_data)

# Prepare data for LSTM
X_train, y_train, scaler = prepare_data(stock_data)

# Train LSTM Model
model = build_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))
model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=1)

# Predict future stock prices (30 & 365 days)
future_days = [30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# ------------------- Plot Results -------------------
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['Close'], mode='lines', name='ICICI Bank Price'
))

# Plot SMA (20-day)
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['SMA_20'], mode='lines', name='20-Day SMA', line=dict(color='blue', dash='dot')
))

# Plot EMA (50-day)
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['EMA_50'], mode='lines', name='50-Day EMA', line=dict(color='red', dash='dot')
))

# Plot RSI
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['RSI'], mode='lines', name='RSI', line=dict(color='purple')
))

# Plot MACD
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['MACD'], mode='lines', name='MACD', line=dict(color='orange')
))
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['MACD_Signal'], mode='lines', name='MACD Signal', line=dict(color='cyan', dash='dot')
))

# Mark Buy Signals
buy_signals = stock_data[stock_data['Signal'] == 'Buy']
fig.add_trace(go.Scatter(
    x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')
))

# Mark Sell Signals
sell_signals = stock_data[stock_data['Signal'] == 'Sell']
fig.add_trace(go.Scatter(
    x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')
))

# Plot Future Predictions
colors = {'30': 'blue', '365': 'red'}
for days in future_days:
    future_dates = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=days)
    fig.add_trace(go.Scatter(
        x=future_dates, y=future_predictions[days],
        mode='lines', name=f'Predicted Prices ({days} Days)', line=dict(dash='dash', color=colors[str(days)])
    ))

fig.update_layout(
    title='ICICI Bank Stock Prediction with Indicators & Buy/Sell Signals',
    xaxis_title='Date', yaxis_title='Stock Price (INR)', template='plotly_dark'
)

fig.show()

# ------------------- Print Predicted Prices -------------------
df_predictions = pd.DataFrame({
    'Date': pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=365),
    'Predicted Price': future_predictions[365]
})
print(df_predictions.head(30))  # First 30 days
print(df_predictions.tail(30))  # Last 30 days (near 1-year mark)


In [ ]:
#MSE (Mean Squared Error)	650.4323	Measures the average squared difference between actual and predicted values. A lower value is better. Since stock prices are in the hundreds/thousands, this suggests moderate error.
#RMSE (Root Mean Squared Error)	25.5036	The square root of MSE. This means, on average, your model's prediction error is around ₹25.50 per stock price.
#MAPE (Mean Absolute Percentage Error)	1.81%	Indicates the model’s error as a percentage of actual stock price. A 1.81% MAPE means your predictions are quite accurate (less than 2% deviation on average).
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Fetch live stock data
def fetch_stock_data(ticker, period='2y', interval='1d'):
    stock = yf.Ticker(ticker)
    data = stock.history(period=period, interval=interval)
    return data[['Close']]

# Prepare data for LSTM
def prepare_data(data, n_steps=60):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']].values)

    X, y = [], []
    for i in range(n_steps, len(scaled_data)):
        X.append(scaled_data[i-n_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    return X, y, scaler

# Build LSTM model
def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Predict future stock prices
def make_future_predictions(model, last_steps, days_to_predict, scaler):
    predictions = []
    for _ in range(days_to_predict):
        pred = model.predict(np.reshape(last_steps, (1, last_steps.shape[0], 1)))
        predictions.append(pred[0, 0])
        last_steps = np.append(last_steps[1:], pred)

    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

# Fetch stock data for ICICI Bank
ticker = 'ICICIBANK.NS'
stock_data = fetch_stock_data(ticker)

# Prepare data for LSTM
X_train, y_train, scaler = prepare_data(stock_data)

# Train LSTM model
model = build_lstm_model(input_shape=(X_train.shape[1], 1))
history = model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)

# Make predictions on training data
y_train_pred = model.predict(X_train)
y_train_pred = scaler.inverse_transform(y_train_pred)
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))

# Calculate accuracy metrics
mse = mean_squared_error(y_train_actual, y_train_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_train_actual, y_train_pred)
mape = np.mean(np.abs((y_train_actual - y_train_pred) / y_train_actual)) * 100

print(f"📈 Model Performance:")
print(f"🔹 Mean Squared Error (MSE): {mse:.4f}")
print(f"🔹 Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"🔹 Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"🔹 R² Score: {r2:.4f}")

# Predict future stock prices (7, 30, 365 days)
future_days = [7, 30, 365]
future_predictions = {days: make_future_predictions(model, X_train[-1], days, scaler) for days in future_days}

# Plot stock prices and predictions
fig = go.Figure()

# Plot actual stock prices
fig.add_trace(go.Scatter(
    x=stock_data.index, y=stock_data['Close'], mode='lines', name='ICICI Bank Price'
))

# Plot future predictions
colors = {'7': 'green', '30': 'blue', '365': 'red'}
for days in future_days:
    future_dates = pd.date_range(stock_data.index[-1] + pd.Timedelta(days=1), periods=days)
    fig.add_trace(go.Scatter(
        x=future_dates, y=future_predictions[days].flatten(),
        mode='lines', name=f'Predicted Prices ({days} Days)', line=dict(dash='dash', color=colors[str(days)])
    ))

fig.update_layout(
    title='ICICI Bank Stock Prediction with Accuracy Metrics',
    xaxis_title='Date', yaxis_title='Stock Price (INR)', template='plotly_dark'
)

fig.show()


In [ ]:
!pip install vaderSentiment

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, LSTM, SimpleRNN
from prettytable import PrettyTable
import yfinance as yf

# Updated list of stocks
stocks = {
    "Large Cap": ["SUNPHARMA.NS", "ALKEM.NS", "LUPIN.NS", "AUROPHARMA.NS"],
    "Mid Cap": ["AJANTPHARM.NS","GLENMARK.NS", "SUVENPHAR.NS"],
    "Small Cap": ["'NEULANDLAB.NS", "MARKSANS.NS'"]
}

# Function to fetch stock data from Yahoo Finance
def get_stock_data(stock, start_date, end_date):
    """
    Fetch live stock data from Yahoo Finance.
    """
    # Fetch stock data using yfinance
    data = yf.download(stock, start=start_date, end=end_date)
    data = data[['Close']].reset_index()  # We are interested in closing prices
    data.rename(columns={'Date': 'date', 'Close': 'price'}, inplace=True)
    return data

# Model architectures
def cnn_model(input_shape):
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
        Flatten(),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def rnn_model(input_shape):
    model = Sequential([
        SimpleRNN(50, activation='relu', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def lstm_model(input_shape):
    model = Sequential([
        LSTM(50, activation='relu', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Prediction function
def predict_november(stock_list, start_date, end_date, prediction_month):
    for stock in stock_list:
        print(f"Predicting stock: {stock}...")
        data = get_stock_data(stock, start_date, end_date)

        if data.empty or len(data) < 100:
            print(f"No sufficient data for stock: {stock}")
            continue

        # Prepare data
        scaler = RobustScaler()
        data['price_scaled'] = scaler.fit_transform(data[['price']])

        look_back = 60
        X, y = [], []
        for i in range(look_back, len(data)):
            X.append(data['price_scaled'].values[i-look_back:i])
            y.append(data['price_scaled'].values[i])

        X, y = np.array(X), np.array(y)
        X = X.reshape((X.shape[0], X.shape[1], 1))

        train_size = int(0.8 * len(X))
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]

        # Train models
        predictions = {}
        maes = {}
        for model_func, model_name in zip([cnn_model, rnn_model, lstm_model], ["CNN", "RNN", "LSTM"]):
            model = model_func(X_train.shape[1:])
            model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

            preds = model.predict(X_test)
            preds_inverse = scaler.inverse_transform(preds)
            actual_inverse = scaler.inverse_transform(y_test.reshape(-1, 1))

            mae = mean_absolute_error(actual_inverse, preds_inverse)
            maes[model_name] = mae
            predictions[model_name] = preds_inverse.flatten()

        # Find best model
        best_model = min(maes, key=maes.get)

        # Generate predictions table for November
        table = PrettyTable()
        table.field_names = ["Date", "Actual", "CNN", "RNN", "LSTM", "CNN+RNN", "CNN+LSTM", "RNN+LSTM", "Ensemble (All)"]

        november_dates = data[data['date'].dt.month == prediction_month]['date']
        november_actual = data[data['date'].dt.month == prediction_month]['price'].values
        november_preds = {
            model: preds[-len(november_dates):].flatten() for model, preds in predictions.items()
        }

        for i, date in enumerate(november_dates):
            cnn, rnn, lstm = november_preds["CNN"][i], november_preds["RNN"][i], november_preds["LSTM"][i]
            cnn_rnn = (cnn + rnn) / 2
            cnn_lstm = (cnn + lstm) / 2
            rnn_lstm = (rnn + lstm) / 2
            ensemble = (cnn + rnn + lstm) / 3
            table.add_row([
                date.strftime('%Y-%m-%d'), november_actual[i], cnn, rnn, lstm,
                cnn_rnn, cnn_lstm, rnn_lstm, ensemble
            ])

        print(f"\n**Stock**: {stock}\n**Best Model**: {best_model} with MAE of {maes[best_model]:.2f}")
        print(table)

# Define date range and prediction month
start_date = "2024-01-01"
end_date = "2024-12-01"
prediction_month = 11  # November

# Run for all automobile stocks
for category, stock_list in stocks.items():
    for stock in stock_list:
        predict_november([stock], start_date, end_date, prediction_month)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, LSTM, SimpleRNN
import yfinance as yf
from openpyxl import Workbook

# Updated list of stocks
stocks = {
    "Large Cap": ["SUNPHARMA.NS", "ALKEM.NS", "LUPIN.NS", "AUROPHARMA.NS"],
    "Mid Cap": ["AJANTPHARM.NS","GLENMARK.NS", "SUVENPHAR.NS"],
    "Small Cap": ["'NEULANDLAB.NS", "MARKSANS.NS'"]
}


# Function to fetch stock data from Yahoo Finance
def get_stock_data(stock, start_date, end_date):
    data = yf.download(stock, start=start_date, end=end_date)
    data = data[['Close']].reset_index()
    data.rename(columns={'Date': 'date', 'Close': 'price'}, inplace=True)
    return data

# Model architectures
def cnn_model(input_shape):
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
        Flatten(),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def rnn_model(input_shape):
    model = Sequential([
        SimpleRNN(50, activation='relu', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def lstm_model(input_shape):
    model = Sequential([
        LSTM(50, activation='relu', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Prediction function
def predict_november(stock_list, start_date, end_date, prediction_month, category_name, writer):
    all_results = []  # To collect all results for this category
    for stock in stock_list:
        print(f"Predicting stock: {stock}...")
        data = get_stock_data(stock, start_date, end_date)

        if data.empty or len(data) < 100:
            print(f"No sufficient data for stock: {stock}")
            continue

        # Prepare data
        scaler = RobustScaler()
        data['price_scaled'] = scaler.fit_transform(data[['price']])

        look_back = 60
        X, y = [], []
        for i in range(look_back, len(data)):
            X.append(data['price_scaled'].values[i-look_back:i])
            y.append(data['price_scaled'].values[i])

        X, y = np.array(X), np.array(y)
        X = X.reshape((X.shape[0], X.shape[1], 1))

        train_size = int(0.8 * len(X))
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]

        # Train models
        predictions = {}
        maes = {}
        for model_func, model_name in zip([cnn_model, rnn_model, lstm_model], ["CNN", "RNN", "LSTM"]):
            model = model_func(X_train.shape[1:])
            model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

            preds = model.predict(X_test)
            preds_inverse = scaler.inverse_transform(preds)
            actual_inverse = scaler.inverse_transform(y_test.reshape(-1, 1))

            mae = mean_absolute_error(actual_inverse, preds_inverse)
            maes[model_name] = mae
            predictions[model_name] = preds_inverse.flatten()

        # Find best model
        best_model = min(maes, key=maes.get)

        # Generate predictions for November
        november_data = data[data['date'].dt.month == prediction_month]
        november_actual = november_data['price'].values
        november_dates = november_data['date'].values

        # Combine predictions
        for i, date in enumerate(november_dates):
            result = {
                "Stock": stock,
                "Date": date,
                "Actual Price": november_actual[i],
                "CNN": predictions["CNN"][-len(november_dates):][i],
                "RNN": predictions["RNN"][-len(november_dates):][i],
                "LSTM": predictions["LSTM"][-len(november_dates):][i],
                "Best Model": best_model,
                "MAE (Best Model)": maes[best_model],
            }
            all_results.append(result)

    # Convert results to DataFrame and write to the specific sheet
    results_df = pd.DataFrame(all_results)
    results_df.to_excel(writer, sheet_name=category_name, index=False)
    print(f"Predictions for {category_name} saved to sheet.")

# Define date range and prediction month
start_date = "2024-01-01"
end_date = "2024-12-01"
prediction_month = 11  # November

# Create an Excel writer
output_file = "Pharmaceutical_Stocks_Predictions_By_Category.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for category, stock_list in stocks.items():
        predict_november(stock_list, start_date, end_date, prediction_month, category, writer)

print(f"Predictions saved to '{output_file}'.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, LSTM, SimpleRNN
import yfinance as yf
from openpyxl import Workbook

# Updated list of stocks (filtering out empty tickers)
stocks = {
    "Large Cap": ["INFY.NS", "TCS.NS", "HCLTECH.NS", "TECHM.NS", "WIPRO.NS"],
    "Mid Cap": ["LTTS.NS", "CTSH"],
    "Small Cap": ["HAPPSTMNDS.NS", "ZENSARTECH.NS"]
}

# Function to fetch stock data from Yahoo Finance
def get_stock_data(stock, start_date, end_date):
    data = yf.download(stock, start=start_date, end=end_date)
    data = data[['Close']].reset_index()
    data.rename(columns={'Date': 'date', 'Close': 'price'}, inplace=True)
    data['date'] = pd.to_datetime(data['date'])  # Ensure 'date' column is datetime
    return data

# Model architectures
def cnn_model(input_shape):
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
        Flatten(),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def rnn_model(input_shape):
    model = Sequential([
        SimpleRNN(50, activation='relu', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def lstm_model(input_shape):
    model = Sequential([
        LSTM(50, activation='relu', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Prediction function
def predict_november(stock_list, start_date, end_date, prediction_month, category_name, writer):
    all_results = []  # To collect all results for this category
    for stock in stock_list:
        print(f"Predicting stock: {stock}...")
        data = get_stock_data(stock, start_date, end_date)

        if data.empty or len(data) < 100:
            print(f"No sufficient data for stock: {stock}")
            continue

        # Prepare data
        scaler = RobustScaler()
        data['price_scaled'] = scaler.fit_transform(data[['price']])

        look_back = 60
        X, y = [], []
        for i in range(look_back, len(data)):
            X.append(data['price_scaled'].values[i-look_back:i])
            y.append(data['price_scaled'].values[i])

        X, y = np.array(X), np.array(y)
        X = X.reshape((X.shape[0], X.shape[1], 1))

        train_size = int(0.8 * len(X))
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]

        # Train models
        predictions = {}
        maes = {}
        for model_func, model_name in zip([cnn_model, rnn_model, lstm_model], ["CNN", "RNN", "LSTM"]):
            model = model_func(X_train.shape[1:])
            model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)  # Increase epochs

            preds = model.predict(X_test)
            preds_inverse = scaler.inverse_transform(preds)
            actual_inverse = scaler.inverse_transform(y_test.reshape(-1, 1))

            mae = mean_absolute_error(actual_inverse, preds_inverse)
            maes[model_name] = mae
            predictions[model_name] = preds_inverse.flatten()

        # Find best model
        best_model = min(maes, key=maes.get)

        # Generate predictions for November
        november_data = data[data['date'].dt.month == prediction_month]
        november_actual = november_data['price'].values
        november_dates = november_data['date'].values

        # Combine predictions
        for i, date in enumerate(november_dates):
            result = {
                "Stock": stock,
                "Date": date,
                "Actual Price": november_actual[i],
                "CNN": predictions["CNN"][-len(november_dates):][i],
                "RNN": predictions["RNN"][-len(november_dates):][i],
                "LSTM": predictions["LSTM"][-len(november_dates):][i],
                "Best Model": best_model,
                "MAE (Best Model)": maes[best_model],
            }
            all_results.append(result)

    # Convert results to DataFrame and write to the specific sheet
    results_df = pd.DataFrame(all_results)
    results_df.to_excel(writer, sheet_name=category_name, index=False)
    print(f"Predictions for {category_name} saved to sheet.")

# Define date range and prediction month
start_date = "2024-01-01"
end_date = "2024-12-01"
prediction_month = 11  # November

# Create an Excel writer
output_file = "IT_Stocks_Predictions_By_Category.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for category, stock_list in stocks.items():
        predict_november(stock_list, start_date, end_date, prediction_month, category, writer)

print(f"Predictions saved to '{output_file}'.")
